In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [ ]:
spark = SparkSession.builder.appName("Airbnb London£").getOrCreate()

In [ ]:
listings = spark.read.csv(
    "listings.csv.gz",
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE",
)
listings.show(5)

In [ ]:
for prop in listings.schema:
    print(prop)

In [ ]:
neighbourhoods = listings.select(listings.neighbourhood_cleansed)
neighbourhoods.show(10, truncate=False)

In [ ]:
high_score_listings = listings \
    .filter((listings.review_scores_location > 4.5) & (listings.number_of_reviews > 100)) \
    .filter(listings.price.isNotNull()) \
    .select("id", "name", "price", "review_scores_location", "number_of_reviews")

high_score_listings.show(20, truncate=False)

In [ ]:
from pyspark.sql.functions import regexp_replace

good_deals_df = high_score_listings.withColumn(
    "price_num", regexp_replace("price", "[$,]", "").cast("float")
) \
.filter(col('price_num') < 100) \
.sort(
    ["price_num", "review_scores_location", "number_of_reviews"],
    ascending=[True, False, False],
)

good_deals_df.schema["price_num"]

In [ ]:
good_deals_df.show(20, truncate=False)

In [ ]:
good_deals_df.count()

In [ ]:
good_deals_df.write.format("csv").save("good_deals.csv")

In [ ]:
spark.stop()